In [1]:
import pandas as pd # import for data reading and manipulation
import re 
from pathlib import Path

%store -r MIMIC_DIR
%store -r cohort
%store -r demo
%store -r comorbid_flags
%store -r mme_summary
%store -r daily_mme_summary
%store -r dose_completeness
%store -r sedative_flag
%store -r lab_summary
%store -r outcomes
%store -r admissions
%store -r palliative_flag
%store -r metastatic_flag
%store -r cancer_type_flags
%store -r overlap_flag
%store -r naloxone_flag
%store -r icu_flag
%store -r vent_flag
%store -r vaso_flag
%store -r sofa
%store -r sofa_component_cols

In [2]:
analysis_df = ( # merging all of the factors for analysis
    cohort
    .merge(demo[["subject_id", "hadm_id", "age_at_admission"]],
           on=["subject_id", "hadm_id"], how="left")
    .merge(comorbid_flags, on="hadm_id", how="left")
    .merge(mme_summary, on="hadm_id", how="left")
    .merge(daily_mme_summary, on="hadm_id", how="left")
    .merge(dose_completeness[["hadm_id", "n_opioid_orders_total", "n_opioid_orders_unconverted", "dose_fully_missing"]],
           on="hadm_id", how="left")
    .merge(sedative_flag, on="hadm_id", how="left")
    .merge(lab_summary, on="hadm_id", how="left")
    .merge(outcomes, on=["subject_id", "hadm_id"], how="left") 
    .merge(admissions[["hadm_id", "insurance", "race", "language"]],
           on= "hadm_id", how = "left")
    .merge(palliative_flag[["hadm_id", "has_palliative_icd", "any_palliative_signal"]],
           on = "hadm_id", how = "left")
    .merge(metastatic_flag, on = "hadm_id", how = "left")
    .merge(cancer_type_flags, on = "hadm_id", how = "left")
    .merge(overlap_flag[["hadm_id", "true_concurrent_use"]], on = "hadm_id", how = "left")
    .merge(naloxone_flag, on = "hadm_id", how = "left")
    .merge(icu_flag, on = "hadm_id", how = "left")
    .merge(vent_flag, on = "hadm_id", how = "left")
    .merge(vaso_flag, on = "hadm_id", how = "left")
    .merge(
        sofa[["hadm_id", "sofa_total", "n_sofa_components_missing"] + sofa_component_cols],
        on = "hadm_id", how = "left"
    )


)


In [3]:
# safety net: coalesce any _x/_y duplicate columns left by a merge instead of hand-picking every time
dupe_bases = sorted({
    c[:-2] for c in analysis_df.columns
    if c.endswith("_x") and c[:-2] + "_y" in analysis_df.columns
})
for base in dupe_bases:
    analysis_df[base] = analysis_df[f"{base}_x"].combine_first(analysis_df[f"{base}_y"])
    analysis_df = analysis_df.drop(columns=[f"{base}_x", f"{base}_y"])

print(f"Coalesced {len(dupe_bases)} duplicate column pairs: {dupe_bases}")

Coalesced 5 duplicate column pairs: ['admittime', 'deathtime', 'dischtime', 'dod', 'hospital_expire_flag']


In [4]:
print(analysis_df["received_naloxone"].isna().sum())
print(analysis_df["received_naloxone"].value_counts(dropna=False))

0
received_naloxone
False    74578
True      1870
Name: count, dtype: int64


In [5]:
# run this before your analysis_df merge to find the duplicate source
print("cohort columns with age:", [c for c in cohort.columns if "age" in c.lower()])
print("demo columns with age:",   [c for c in demo.columns if "age" in c.lower()])

# check every dataframe being merged
for name, df in [("comorbid_flags", comorbid_flags), 
                  ("mme_summary", mme_summary),
                  ("outcomes", outcomes),
                  ("lab_summary", lab_summary)]:
    age_cols = [c for c in df.columns if "age" in c.lower()]
    if age_cols:
        print(f"⚠️  {name} also has age columns: {age_cols}")

cohort columns with age: ['anchor_age']
demo columns with age: ['anchor_age', 'age_at_admission']


In [6]:
# FILLNA TYPE
bool_cols = [
    "concurrent_sedative_use",
    "has_palliative_icd",
    "any_palliative_signal",
    "has_metastatic_disease",
    "true_concurrent_use",
    "received_naloxone",
    "had_icu_stay",
    "mechanically_ventilated",
    "received_vasopressors",
    "dose_fully_missing",
] + [c for c in analysis_df.columns if c.startswith("cancer_")]

analysis_df[bool_cols] = analysis_df[bool_cols].fillna(False)

count_cols = [
    "n_sedative_orders",
    "n_naloxone_orders",
    "n_icu_stays",
    "total_icu_los_days",
    "n_vent_events",
    "n_vasopressor_events",
    "n_opioid_orders",
    "n_opioid_orders_total",
    "n_opioid_orders_unconverted",
    "n_cancer_types_flagged",
] 
analysis_df[count_cols] = analysis_df[count_cols].fillna(0)

analysis_df["renal_impairment"] = analysis_df["max_Creatinine"] > 1.5
analysis_df["hepatic_impairment"] = analysis_df["max_Bilirubin, Total"] > 2.0


In [7]:
print(f"Final cohort shape: {analysis_df.shape}")
print(f"\nMissing value summary (>0):")
missing = analysis_df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))
print(f"\nKey feature rates: ")
print(f"  ICU stay:              {analysis_df.had_icu_stay.mean():.1%}")
print(f"  Mechanically vented:   {analysis_df.mechanically_ventilated.mean():.1%}")
print(f"  Vasopressors:          {analysis_df.received_vasopressors.mean():.1%}")
print(f"  Naloxone given:        {analysis_df.received_naloxone.mean():.1%}")
print(f"  True concurrent use:   {analysis_df.true_concurrent_use.mean():.1%}")
print(f"  Palliative signal:     {analysis_df.any_palliative_signal.mean():.1%}")
print(f"  Metastatic:            {analysis_df.has_metastatic_disease.mean():.1%}")
print(f"  Renal impairment:      {analysis_df.renal_impairment.mean():.1%}")
print(f"  Hepatic impairment:    {analysis_df.hepatic_impairment.mean():.1%}")
print(f"  Died within 48h:       {analysis_df.died_within_48h.mean():.1%}")
print(f"  Exposed, dose unknown: {analysis_df.dose_fully_missing.mean():.1%}  (opioid order present, but 0 convertible to an MME dose)")

analysis_df.to_parquet("analysis_ready_cohort_df.parquet")
print(f"Final analysis-ready cohort shape: {len(analysis_df)}")

Final cohort shape: (76448, 80)

Missing value summary (>0):
hours_to_death                          72969
deathtime                               72969
sofa_respiration                        70358
sofa_liver                              69063
sofa_coagulation                        64048
sofa_cns                                63706
sofa_total                              63645
n_sofa_components_missing               63645
sofa_renal                              63645
sofa_cardiovascular                     63645
dod                                     32608
mean_daily_mme                          32535
max_daily_mme                           32535
total_mme                               32533
max_single_dose_mme                     32533
mean_Bilirubin, Total                   30504
max_Bilirubin, Total                    30504
max_Alanine Aminotransferase (ALT)      30236
mean_Alanine Aminotransferase (ALT)     30236
max_Asparate Aminotransferase (AST)     30006
mean_Asparate Amino

In [8]:
analysis_df = pd.read_parquet("analysis_ready_cohort_df.parquet")
analysis_df.to_csv("analysis_ready_cohort_df.csv", index = False)
print(f"Saved, Shape: {analysis_df.shape}")

Saved, Shape: (76448, 80)


In [9]:
print(f"Number of features (columns): {analysis_df.shape[1]}")
print(f"Number of rows: {analysis_df.shape[0]}")

# see all feature names
print("\nAll features:")
for i, col in enumerate(analysis_df.columns):
    print(f"  {i+1:>3}. {col}")

Number of features (columns): 80
Number of rows: 76448

All features:
    1. subject_id
    2. hadm_id
    3. icd_codes
    4. anchor_age
    5. anchor_year
    6. gender
    7. exposed
    8. admit_year
    9. age_at_admission
   10. has_diabetes
   11. has_renal_disease
   12. has_liver_disease
   13. has_copd
   14. has_substance_use_disorder
   15. total_mme
   16. n_opioid_orders
   17. max_single_dose_mme
   18. mean_daily_mme
   19. max_daily_mme
   20. n_opioid_orders_total
   21. n_opioid_orders_unconverted
   22. dose_fully_missing
   23. n_sedative_orders
   24. concurrent_sedative_use
   25. mean_Alanine Aminotransferase (ALT)
   26. mean_Asparate Aminotransferase (AST)
   27. mean_Bilirubin, Total
   28. mean_Creatinine
   29. max_Alanine Aminotransferase (ALT)
   30. max_Asparate Aminotransferase (AST)
   31. max_Bilirubin, Total
   32. max_Creatinine
   33. los_days
   34. died_within_30d_discharge
   35. hours_to_death
   36. died_within_48h
   37. insurance
   38. race

In [10]:
print(f" Before age filter: {len(analysis_df):,}")
analysis_df = analysis_df[analysis_df["anchor_age"] >= 18].copy().reset_index(drop=True)
print(f"shape after age filter: {analysis_df.shape}")
print(f"Index runs 0 to {analysis_df.index.max()}")
print(f"After age filter: {len(analysis_df):,}")

 Before age filter: 76,448
shape after age filter: (76448, 80)
Index runs 0 to 76447
After age filter: 76,448


In [11]:
analysis_df.to_parquet("analysis_ready_cohort_df_over_18.parquet")
analysis_df = pd.read_parquet("analysis_ready_cohort_df_over_18.parquet")
analysis_df.to_csv("analysis_ready_cohort_df_over_18.csv", index = False)
print(f"Saved, Shape: {analysis_df.shape}")

Saved, Shape: (76448, 80)


In [12]:
print(f"Min age: {analysis_df.anchor_age.min()}")  # should be 18
print(f"Max age: {analysis_df.anchor_age.max()}")
print(f"Any under 18: {(analysis_df.anchor_age < 18).sum()}")  # should be 0

Min age: 18
Max age: 91
Any under 18: 0


In [13]:
print([c for c in analysis_df.columns if "age" in c.lower()])


['anchor_age', 'age_at_admission', 'language']


In [14]:
# check for any other duplicated columns
x_cols = [c for c in analysis_df.columns if c.endswith("_x")]
print("Duplicated columns:", [c.replace("_x", "") for c in x_cols])

Duplicated columns: []


In [15]:
# ── SANITY CHECK ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

analysis_df = pd.read_parquet("analysis_ready_cohort_df.parquet")

print("=" * 60)
print("1. BASIC SHAPE")
print("=" * 60)
print(f"Rows:    {analysis_df.shape[0]:,}")
print(f"Columns: {analysis_df.shape[1]:,}")
# expect 253,524 rows from your earlier output


print("\n" + "=" * 60)
print("2. DUPLICATE CHECK")
print("=" * 60)
print(f"Duplicate hadm_id rows:    {analysis_df.hadm_id.duplicated().sum():,}")
print(f"Duplicate subject_id rows: {analysis_df.subject_id.duplicated().sum():,}")
# hadm_id should be 0 — each admission is one row
# subject_id duplicates are expected (patients can have multiple admissions)


print("\n" + "=" * 60)
print("3. MISSING VALUES")
print("=" * 60)
missing = analysis_df.isnull().sum()
missing_pct = (missing / len(analysis_df) * 100).round(1)
missing_df = pd.DataFrame({
    "missing_n": missing,
    "missing_%": missing_pct
}).query("missing_n > 0").sort_values("missing_%", ascending=False)

if len(missing_df):
    print(missing_df.to_string())
else:
    print("No missing values.")
# flag anything >20% missing as worth investigating
high_missing = missing_df[missing_df["missing_%"] > 20]
if len(high_missing):
    print(f"\n⚠️  {len(high_missing)} columns >20% missing:")
    print(high_missing.to_string())


print("\n" + "=" * 60)
print("4. BOOLEAN FLAG SANITY")
print("=" * 60)
bool_cols = analysis_df.select_dtypes(include="bool").columns.tolist()
for col in bool_cols:
    rate = analysis_df[col].mean()
    flag = "⚠️ " if rate < 0.001 or rate > 0.999 else "  "
    print(f"{flag}{col:<45} {rate:.1%}")
# flag anything suspiciously close to 0% or 100%


print("\n" + "=" * 60)
print("5. CLINICAL PLAUSIBILITY")
print("=" * 60)

# age
print(f"Age range:        {analysis_df.anchor_age.min():.0f} – {analysis_df.anchor_age.max():.0f}")
print(f"Age mean (SD):    {analysis_df.anchor_age.mean():.1f} ({analysis_df.anchor_age.std():.1f})")
# expect roughly 50-80 for a cancer cohort

# length of stay
print(f"\nLOS range:        {analysis_df.los_days.min():.1f} – {analysis_df.los_days.max():.1f} days")
print(f"LOS median:       {analysis_df.los_days.median():.1f} days")
print(f"Negative LOS:     {(analysis_df.los_days < 0).sum()}")  # should be 0

# MME
print(f"\nTotal MME range:  {analysis_df.total_mme.min():.1f} – {analysis_df.total_mme.max():.1f}")
print(f"Total MME median: {analysis_df.total_mme.median():.1f}")
extreme_mme = (analysis_df.total_mme > 10_000).sum()
print(f"Admissions >10k MME: {extreme_mme:,}  ({extreme_mme/len(analysis_df):.1%})")
# very high MME warrants review — could be data entry errors

# SOFA
if "sofa_total" in analysis_df.columns:
    print(f"\nSOFA range:       {analysis_df.sofa_total.min():.0f} – {analysis_df.sofa_total.max():.0f}")
    print(f"SOFA mean (SD):   {analysis_df.sofa_total.mean():.1f} ({analysis_df.sofa_total.std():.1f})")
    print(f"SOFA missing:     {analysis_df.sofa_total.isna().sum():,} ({analysis_df.sofa_total.isna().mean():.1%})")
    # SOFA only populated for ICU patients so missing for non-ICU is expected


print("\n" + "=" * 60)
print("6. LOGICAL CONSISTENCY CHECKS")
print("=" * 60)

# naloxone without opioids — suspicious
if "received_naloxone" in analysis_df.columns and "n_opioid_orders" in analysis_df.columns:
    naloxone_no_opioids = (
        analysis_df["received_naloxone"] & (analysis_df["n_opioid_orders"] == 0)
    ).sum()
    print(f"Naloxone given but no opioid orders: {naloxone_no_opioids:,}")
    # should be near 0 — if high, your naloxone cohort filter may be too broad

# ICU but no SOFA — expected for non-icu, but all ICU patients should have some sofa data
if "had_icu_stay" in analysis_df.columns and "sofa_total" in analysis_df.columns:
    icu_no_sofa = (analysis_df["had_icu_stay"] & analysis_df["sofa_total"].isna()).sum()
    print(f"ICU stay but no SOFA score:          {icu_no_sofa:,}")

# ventilated but no ICU stay — shouldn't happen
if "mechanically_ventilated" in analysis_df.columns and "had_icu_stay" in analysis_df.columns:
    vent_no_icu = (
        analysis_df["mechanically_ventilated"] & ~analysis_df["had_icu_stay"]
    ).sum()
    print(f"Ventilated but no ICU stay:          {vent_no_icu:,}")
    # should be 0 or very small

# metastatic but no cancer type flagged
cancer_cols = [c for c in analysis_df.columns if c.startswith("cancer_")]
if "has_metastatic_disease" in analysis_df.columns and cancer_cols:
    meta_no_cancer = (
        analysis_df["has_metastatic_disease"] & ~analysis_df[cancer_cols].any(axis=1)
    ).sum()
    print(f"Metastatic but no cancer type flag:  {meta_no_cancer:,}")
    # some expected — metastatic_unspecified codes may not map to a primary type

# died in hospital but has 30-day post-discharge mortality — impossible
if "hospital_expire_flag" in analysis_df.columns and "died_within_30d_discharge" in analysis_df.columns:
    dead_and_discharged = (
        (analysis_df["hospital_expire_flag"] == 1) &
        analysis_df["died_within_30d_discharge"]
    ).sum()
    print(f"Died in hospital + 30d post-discharge death: {dead_and_discharged:,}")
    # should be 0


print("\n" + "=" * 60)
print("7. KEY RATE SUMMARY")
print("=" * 60)
rates = {
    "Female":                   (analysis_df.gender == "F").mean(),
    "Metastatic disease":        analysis_df.get("has_metastatic_disease", pd.Series([None])).mean(),
    "Any palliative signal":     analysis_df.get("any_palliative_signal", pd.Series([None])).mean(),
    "Concurrent sedative use":   analysis_df.get("concurrent_sedative_use", pd.Series([None])).mean(),
    "True concurrent overlap":   analysis_df.get("true_concurrent_use", pd.Series([None])).mean(),
    "Received naloxone":         analysis_df.get("received_naloxone", pd.Series([None])).mean(),
    "ICU stay":                  analysis_df.get("had_icu_stay", pd.Series([None])).mean(),
    "Mechanically ventilated":   analysis_df.get("mechanically_ventilated", pd.Series([None])).mean(),
    "Vasopressors":              analysis_df.get("received_vasopressors", pd.Series([None])).mean(),
    "Renal impairment":          analysis_df.get("renal_impairment", pd.Series([None])).mean(),
    "Hepatic impairment":        analysis_df.get("hepatic_impairment", pd.Series([None])).mean(),
    "Hospital mortality":        (analysis_df.get("hospital_expire_flag", pd.Series([0])) == 1).mean(),
    "30-day post-discharge death": analysis_df.get("died_within_30d_discharge", pd.Series([None])).mean(),
}
for label, rate in rates.items():
    if pd.notna(rate):
        print(f"  {label:<40} {rate:.1%}")

print("\n✅ Sanity check complete.")

1. BASIC SHAPE
Rows:    76,448
Columns: 80

2. DUPLICATE CHECK
Duplicate hadm_id rows:    0
Duplicate subject_id rows: 46,676

3. MISSING VALUES
                                      missing_n  missing_%
hours_to_death                            72969       95.4
deathtime                                 72969       95.4
sofa_respiration                          70358       92.0
sofa_liver                                69063       90.3
sofa_coagulation                          64048       83.8
sofa_cardiovascular                       63645       83.3
sofa_total                                63645       83.3
n_sofa_components_missing                 63645       83.3
sofa_renal                                63645       83.3
sofa_cns                                  63706       83.3
dod                                       32608       42.7
total_mme                                 32533       42.6
max_single_dose_mme                       32533       42.6
mean_daily_mme               

In [16]:
# ── SANITY CHECK ──────────────────────────────────────────────────────────────

print("=" * 60)
print("1. BASIC SHAPE")
print("=" * 60)
print(f"Rows:    {analysis_df.shape[0]:,}")
print(f"Columns: {analysis_df.shape[1]:,}")


print("\n" + "=" * 60)
print("2. DUPLICATE CHECK")
print("=" * 60)
print(f"Duplicate hadm_id rows:    {analysis_df.hadm_id.duplicated().sum():,}")
print(f"Duplicate subject_id rows: {analysis_df.subject_id.duplicated().sum():,}")


print("\n" + "=" * 60)
print("3. MISSING VALUES")
print("=" * 60)
missing = analysis_df.isnull().sum()
missing_pct = (missing / len(analysis_df) * 100).round(1)
missing_df = pd.DataFrame({
    "missing_n": missing,
    "missing_%": missing_pct
}).query("missing_n > 0").sort_values("missing_%", ascending=False)

if len(missing_df):
    print(missing_df.to_string())
else:
    print("No missing values.")

high_missing = missing_df[missing_df["missing_%"] > 20]
if len(high_missing):
    print(f"\n⚠️  {len(high_missing)} columns >20% missing:")
    print(high_missing.to_string())


print("\n" + "=" * 60)
print("4. BOOLEAN FLAG SANITY")
print("=" * 60)
bool_cols = analysis_df.select_dtypes(include="bool").columns.tolist()
for col in bool_cols:
    rate = analysis_df[col].mean()
    flag = "⚠️ " if rate < 0.001 or rate > 0.999 else "  "
    print(f"{flag}{col:<45} {rate:.1%}")


print("\n" + "=" * 60)
print("5. CLINICAL PLAUSIBILITY")
print("=" * 60)

# age
print(f"Age range:        {analysis_df.anchor_age.min():.0f} – {analysis_df.anchor_age.max():.0f}")
print(f"Age mean (SD):    {analysis_df.anchor_age.mean():.1f} ({analysis_df.anchor_age.std():.1f})")
print(f"Any under 18:     {(analysis_df.anchor_age < 18).sum():,}")  # should be 0

# length of stay
print(f"\nLOS range:        {analysis_df.los_days.min():.1f} – {analysis_df.los_days.max():.1f} days")
print(f"LOS median:       {analysis_df.los_days.median():.1f} days")
print(f"Negative LOS:     {(analysis_df.los_days < 0).sum():,}")  # should be 0

# MME
print(f"\nTotal MME range:  {analysis_df.total_mme.min():.1f} – {analysis_df.total_mme.max():.1f}")
print(f"Total MME median: {analysis_df.total_mme.median():.1f}")
extreme_mme = (analysis_df.total_mme > 10_000).sum()
print(f"Admissions >10k MME: {extreme_mme:,}  ({extreme_mme/len(analysis_df):.1%})")


print("\n" + "=" * 60)
print("6. LOGICAL CONSISTENCY CHECKS")
print("=" * 60)

# naloxone without opioids
if "received_naloxone" in analysis_df.columns and "n_opioid_orders" in analysis_df.columns:
    naloxone_no_opioids = (
        analysis_df["received_naloxone"] & (analysis_df["n_opioid_orders"] == 0)
    ).sum()
    print(f"Naloxone given but no opioid orders:         {naloxone_no_opioids:,}")

# ventilated but no ICU stay
if "mechanically_ventilated" in analysis_df.columns and "had_icu_stay" in analysis_df.columns:
    vent_no_icu = (
        analysis_df["mechanically_ventilated"] & ~analysis_df["had_icu_stay"]
    ).sum()
    print(f"Ventilated but no ICU stay:                  {vent_no_icu:,}")

# metastatic but no cancer type flagged
cancer_cols = [c for c in analysis_df.columns if c.startswith("cancer_")]
if "has_metastatic_disease" in analysis_df.columns and cancer_cols:
    meta_no_cancer = (
        analysis_df["has_metastatic_disease"] & ~analysis_df[cancer_cols].any(axis=1)
    ).sum()
    print(f"Metastatic but no cancer type flag:          {meta_no_cancer:,}")

# died in hospital but also has 30 day post discharge death
if "hospital_expire_flag" in analysis_df.columns and "died_within_30d_discharge" in analysis_df.columns:
    dead_and_discharged = (
        (analysis_df["hospital_expire_flag"] == 1) &
        analysis_df["died_within_30d_discharge"]
    ).sum()
    print(f"Died in hospital + 30d post-discharge death: {dead_and_discharged:,}")

# renal impairment but no creatinine value
creat_col = [c for c in analysis_df.columns if "creat" in c.lower() and "max" in c.lower()]
if creat_col and "renal_impairment" in analysis_df.columns:
    renal_no_creat = (
        analysis_df["renal_impairment"] & analysis_df[creat_col[0]].isna()
    ).sum()
    print(f"Renal impairment but no creatinine value:    {renal_no_creat:,}")


print("\n" + "=" * 60)
print("7. KEY RATE SUMMARY")
print("=" * 60)
rates = {
    "Female":                        (analysis_df.gender == "F").mean(),
    "Metastatic disease":             analysis_df.get("has_metastatic_disease", pd.Series([None])).mean(),
    "Any palliative signal":          analysis_df.get("any_palliative_signal", pd.Series([None])).mean(),
    "Concurrent sedative use":        analysis_df.get("concurrent_sedative_use", pd.Series([None])).mean(),
    "True concurrent overlap":        analysis_df.get("true_concurrent_use", pd.Series([None])).mean(),
    "Received naloxone":              analysis_df.get("received_naloxone", pd.Series([None])).mean(),
    "ICU stay":                       analysis_df.get("had_icu_stay", pd.Series([None])).mean(),
    "Mechanically ventilated":        analysis_df.get("mechanically_ventilated", pd.Series([None])).mean(),
    "Vasopressors":                   analysis_df.get("received_vasopressors", pd.Series([None])).mean(),
    "Renal impairment":               analysis_df.get("renal_impairment", pd.Series([None])).mean(),
    "Hepatic impairment":             analysis_df.get("hepatic_impairment", pd.Series([None])).mean(),
    "Hospital mortality":             (analysis_df.get("hospital_expire_flag", pd.Series([0])) == 1).mean(),
    "30-day post-discharge death":    analysis_df.get("died_within_30d_discharge", pd.Series([None])).mean(),
}
for label, rate in rates.items():
    if pd.notna(rate):
        print(f"  {label:<40} {rate:.1%}")

print("\n✅ Sanity check complete.")

1. BASIC SHAPE
Rows:    76,448
Columns: 80

2. DUPLICATE CHECK
Duplicate hadm_id rows:    0
Duplicate subject_id rows: 46,676

3. MISSING VALUES
                                      missing_n  missing_%
hours_to_death                            72969       95.4
deathtime                                 72969       95.4
sofa_respiration                          70358       92.0
sofa_liver                                69063       90.3
sofa_coagulation                          64048       83.8
sofa_cardiovascular                       63645       83.3
sofa_total                                63645       83.3
n_sofa_components_missing                 63645       83.3
sofa_renal                                63645       83.3
sofa_cns                                  63706       83.3
dod                                       32608       42.7
total_mme                                 32533       42.6
max_single_dose_mme                       32533       42.6
mean_daily_mme               

In [17]:
missing = analysis_df.isnull().sum()
missing_pct = (missing / len(analysis_df) * 100).round(1)
missing_df = pd.DataFrame({
    "missing_n": missing,
    "missing_%": missing_pct
}).query("missing_n > 0").sort_values("missing_%", ascending=False)
print(missing_df.to_string())

                                      missing_n  missing_%
hours_to_death                            72969       95.4
deathtime                                 72969       95.4
sofa_respiration                          70358       92.0
sofa_liver                                69063       90.3
sofa_coagulation                          64048       83.8
sofa_cardiovascular                       63645       83.3
sofa_total                                63645       83.3
n_sofa_components_missing                 63645       83.3
sofa_renal                                63645       83.3
sofa_cns                                  63706       83.3
dod                                       32608       42.7
total_mme                                 32533       42.6
max_single_dose_mme                       32533       42.6
mean_daily_mme                            32535       42.6
max_daily_mme                             32535       42.6
mean_Bilirubin, Total                     30504       39

In [18]:
# if any bool flags are still missing, fillna was skipped or ran before the merge
bool_cols = [
    "concurrent_sedative_use", "has_palliative_icd",
    "any_palliative_signal", "has_metastatic_disease", "true_concurrent_use",
    "received_naloxone", "had_icu_stay", "mechanically_ventilated",
    "received_vasopressors",
] + [c for c in analysis_df.columns if c.startswith("cancer_")]

analysis_df[bool_cols] = analysis_df[bool_cols].fillna(False)


In [19]:
count_cols = [
    "n_sedative_orders", "n_naloxone_orders", "n_icu_stays",
    "total_icu_los_days", "n_vent_events", "n_vasopressor_events",
    "n_opioid_orders", "n_cancer_types_flagged",
]
# only fill columns that actually exist
count_cols_present = [c for c in count_cols if c in analysis_df.columns]
analysis_df[count_cols_present] = analysis_df[count_cols_present].fillna(0)

In [20]:
# step 1 — check mme_summary exists and looks right
print(mme_summary.shape)
print(mme_summary.head())
print(f"mme_summary hadm_ids: {mme_summary.hadm_id.nunique():,}")

# step 2 — check overlap between mme_summary and analysis_df
mme_ids = set(mme_summary.hadm_id)
cohort_ids = set(analysis_df.hadm_id)
print(f"\nCohort hadm_ids:          {len(cohort_ids):,}")
print(f"MME hadm_ids:             {len(mme_ids):,}")
print(f"Overlap:                  {len(mme_ids & cohort_ids):,}")
print(f"In cohort but not MME:    {len(cohort_ids - mme_ids):,}")
# "in cohort but not MME" is patients with NO opioid orders — this is fine
# if overlap is very low that means a key mismatch

(43915, 4)
    hadm_id  total_mme  n_opioid_orders  max_single_dose_mme
0  20000045    23044.0                6               5760.0
1  20000629        5.0                1                  5.0
2  20001002        8.0                1                  8.0
3  20001259       45.0                1                 45.0
4  20001371      231.5               15                 80.0
mme_summary hadm_ids: 43,915

Cohort hadm_ids:          76,448
MME hadm_ids:             43,915
Overlap:                  43,915
In cohort but not MME:    32,533
